# Makemore Part 2 · Stage 1 Recall

Use this notebook for Stage 1 of the `karpathy-review-sop`.

Rules:

- Write from memory first.
- Do not look up concepts.
- Karpathy notebook may be used only as a dictionary for API or syntax.
- Mark unclear points with `?` instead of stalling.
- Use hints only to get started, not to replace recall.

## Session Info

- Date:
- Lesson:
- Start time:
- Stop time:

## 1. Problem Definition

Write what problem this lesson is solving.

Hints:

- What is one training example?
- What is the input: one char, or a fixed-length context?
- What is the output: a whole word, or the next char?
- Why is Part 1 not enough?

Template:

- Input: 3 characters a p
  air, make the chars into embedding index as the input in the network
- Output: the next output based on the input index, and itos to predict the next cha based on the first one
- Training goal: the goal is to make the network output the chas are close to our predict output, the training and dev split data loss are as low as possible, and their values are very similar. 
- Why this is an upgrade over Part 1: the diagram can only contain 2 chars, the dimension will be huge if there are more data.

---

### Stage 2 Feedback

<p style="color:#14823b"><strong>Green:</strong> You correctly identified that Part 2 uses a fixed-length context and feeds it into a neural network through embeddings.</p>

<p style="color:#b42318"><strong>Red:</strong> Output is not <code>itos to predict the next char based on the first one</code>.</p>

Correct version: given a fixed-length context, the model predicts the next character index. <code>itos</code> is only used later to convert an index back into a readable character.

<p style="color:#b42318"><strong>Red:</strong> Part 1 is not limited because the diagram can only contain 2 chars.</p>

Correct version: Part 1 bigram only conditions on one previous character. Extending raw count tables to longer contexts creates a huge sparse table, so Part 2 uses embeddings plus an MLP to generalize across contexts.


## 2. Math / Loss Formula

Write the core math objects and the loss used in this lesson.

Hints:

- What are `C`, `W1`, `b1`, `W2`, `b2`?
- What is `logits`?
- What does `F.cross_entropy(logits, Y)` compare?
- If you remember NLL form, write it too.

Template:

- Parameters:

C = torch.randn((27,2)) # embedding the input for 2 dimention, but we can also change the dim number for dicreasing the loss

W1: the weights in the hidden layer

b1: the bias in the hidden layer

W2: the weights in the output layer

b2: the bias in the hidden layer

- Forward objects:

h = C[X].view(-1, 6) @ W1 + b1

logits = h @ W2 + b2

- Loss:

loss = F.cross_entropy(logits, Y)

counts = logits.exp()

prob = counts / counts.sum(1, keepdim=True)

loss = -probs[arange(32), Y].log().mean()

- What the loss is trying to make happen:

compare the output from the network and the predict output we want the network to make

---

### Stage 2 Feedback

<p style="color:#14823b"><strong>Green:</strong> You remembered the main parameter groups: <code>C</code>, <code>W1</code>, <code>b1</code>, <code>W2</code>, <code>b2</code>.</p>

<p style="color:#b42318"><strong>Red:</strong> Hidden layer formula is missing <code>tanh</code>.</p>

```python
h = torch.tanh(C[X].view(-1, 6) @ W1 + b1)
```

<p style="color:#b42318"><strong>Red:</strong> <code>b2</code> is not hidden layer bias.</p>

Correct version: <code>b1</code> is hidden layer bias; <code>b2</code> is output layer bias.

<p style="color:#b42318"><strong>Red:</strong> Manual NLL variable names are inconsistent.</p>

```python
counts = logits.exp()
prob = counts / counts.sum(1, keepdim=True)
loss = -prob[torch.arange(batch_size), Y].log().mean()
```

<p style="color:#b42318"><strong>Red:</strong> <code>torch.arange(32)</code> only works for a mini-batch of exactly 32 examples.</p>

Correct version: use <code>torch.arange(batch_size)</code> for mini-batches, or <code>torch.arange(Y.shape[0])</code> for the full dataset.


## 3. Algorithm Skeleton / Pseudocode

Write the training and generation skeleton in plain language or pseudocode.

Hints:

- Training skeleton should include: sample batch -> embedding lookup -> hidden layer -> logits -> loss -> zero grad -> backward -> update.
- Generation skeleton should include: start context -> predict next char -> sample -> roll context -> stop on end token.

Template:

```text
training:

for _ in range(10000):

    xi = (32 random index from X)

    emb = C[X[xi]]

    h = emb.view(-1, 6) @ W1 + b1

    logits = h @ W2 + b2

    loss = F.cross_entropy(logits, Yi)

    for p in parameters:

        p.grad = None

    loss.backward()

    p.data += -0.1 * p.grad

print(loss.item())
    

generation:


for _ in range (10):

    out = []

    
    context = [0] * block_size

    while Ture:

        emb = C[torch.tensor([context])]

        h = torch.tanh(emb * W1 + b1)

        logits = h * W2 + b2

        probs = F.softmax(logits, dim = 1)

        xi = torch.multinomial(probs, num_sample=1, generator = g.item())

        context = context[1:] + [xi]

        out.append[xi]


        if xi == 0:

            break
```

---

### Stage 2 Feedback

<p style="color:#14823b"><strong>Green:</strong> Your training spine has the right order: sample batch -> embedding lookup -> hidden layer -> logits -> loss -> zero grad -> backward -> update.</p>

<p style="color:#b42318"><strong>Red:</strong> Target should be indexed by the sampled mini-batch indices.</p>

```python
loss = F.cross_entropy(logits, Y[ix])
```

<p style="color:#b42318"><strong>Red:</strong> Parameter update must happen inside the parameter loop.</p>

```python
for p in parameters:
    p.data += -lr * p.grad
```

<p style="color:#b42318"><strong>Red:</strong> Hidden layer should include <code>tanh</code>.</p>

```python
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
```

<p style="color:#b42318"><strong>Red:</strong> In generation, use matrix multiplication <code>@</code>, not elementwise multiplication <code>*</code>, for the layers.</p>

```python
emb = C[torch.tensor([context])]
h = torch.tanh(emb.view(1, -1) @ W1 + b1)
logits = h @ W2 + b2
```

<p style="color:#b42318"><strong>Red:</strong> <code>torch.multinomial</code> arguments and output handling need correction.</p>

```python
probs = F.softmax(logits, dim=1)
ix = torch.multinomial(probs, num_samples=1, generator=g).item()
```

<p style="color:#b42318"><strong>Red:</strong> Context update and output append should use a plain Python integer index.</p>

```python
context = context[1:] + [ix]
out.append(ix)
if ix == 0:
    break
```

<p style="color:#b42318"><strong>Red:</strong> Typo-level but important when coding from memory: <code>while True</code>, not <code>while Ture</code>; <code>out.append(ix)</code>, not <code>out.append[ix]</code>.</p>


## 4. Interface With Part 1

Write how this lesson connects to the previous lesson.

Hints:

- What stayed the same from Part 1?
- What changed in the input representation?
- How is this still a next-character prediction task?
- What did bigram do that this model now generalizes?

Template:

- Same as Part 1: still can use backward and propagation to reduce the loss, the fundation concept is the same
- New in Part 2: the input representation is change to be 3 blocks, and the embedding dim can be bigger than part1
- Why this matters: biagram only read the first cha, there will be a huge count table if there are many chars

---

### Stage 2 Feedback

<p style="color:#14823b"><strong>Green:</strong> You correctly remembered that Part 2 is still next-character prediction and still uses gradient descent/backprop to reduce loss.</p>

<p style="color:#14823b"><strong>Green:</strong> You correctly identified the important upgrade: multiple context characters plus embeddings.</p>

<p style="color:#b42318"><strong>Red:</strong> Write <code>bigram</code>, not <code>biagram</code>, and say it reads the previous character, not the first character.</p>

Correct version: bigram conditions on one previous character. The MLP generalizes this by conditioning on a fixed-length context, such as three previous characters, without building a huge raw count table for every possible context.


## 5. My `?` List

Write every place where your recall breaks.

Good `?` examples:

- `? why emb.view(-1, 6)`
- `? why block_size = 3`
- `? why requires_grad must be True`
- `? cross_entropy vs manual NLL`

- ?why test split can't be use multiple times
- ? why we need 3 splits in data

---

### Stage 2 Feedback

<p style="color:#14823b"><strong>Green:</strong> These are useful <code>?</code> items. They point to real generalization questions, not just syntax gaps.</p>

<p style="color:#b42318"><strong>Red:</strong> Test split should not be used repeatedly for model decisions.</p>

Correct version: train is for fitting parameters; dev/val is for choosing hyperparameters; test is used once at the end for final generalization estimate.


## Optional: Shapes I Think I Remember

If helpful, write tensor shapes from memory.

Hints:

- `C`
- `emb`
- `emb.view(...)`
- `h`
- `logits`

x: (N, 3)
Y: (N,)
C: (27, 2)
emb: (batch_size, 3, 2)
emb.view(-1, 6): (batch_size, 6)
h: (batch_size, hidden_size)
logits: (batch_size, 27)

---

### Stage 2 Feedback

<p style="color:#14823b"><strong>Green:</strong> Your shape spine is correct for <code>embedding_dim = 2</code> and <code>block_size = 3</code>.</p>

<p style="color:#b42318"><strong>Red:</strong> If <code>embedding_dim = 10</code>, then every <code>6</code> that came from <code>3 * 2</code> must become <code>30</code>.</p>

```text
C: (27, 10)
emb: (batch_size, 3, 10)
emb.view(-1, 30): (batch_size, 30)
W1: (30, hidden_size)
```
